In [6]:
import torch
import numpy as np
from argparse import ArgumentParser
from tqdm import tqdm

from utils import *
from src.iipw import *

d1 = 10000
d2 = 1000
r = 10
p = 5 / d2
ob = 10
sample = "iid"
if torch.cuda.is_available():
    device = 'cuda:1'
else:
    device = 'cpu'

dataset = "syn"

M = load_normalized_data_syn(r, d1, d2, device)

p_hat_count = torch.zeros((d2, d2)).to(device)
for run in range(1):    
    if sample == "iid":
        # IID sample
        observed_M, masks = get_uniform_masks(M, p)
    else:
        # few entry sample
        observed_M, masks = get_random_samples_per_row(M.cpu().numpy(), ob)
        p = ob/d2
        observed_M = torch.from_numpy(observed_M).float().to(device)
        masks = torch.from_numpy(masks).to(device)
    # observed second-moment matrix
    MTM = M.T @ M
    normalized_MTM = MTM / d1
    second_moment_observe_M =  observed_M.T @ observed_M
    T_masks = 1 * (second_moment_observe_M!=0)

    second_moment_observe_M =  observed_M.T @ observed_M
    second_moment_observe_count = (1 * (observed_M != 0)).float().T @ (1 * (observed_M != 0).float())
    p_hat_count = p_hat_count + second_moment_observe_count
    second_moment_observe_count = second_moment_observe_count + (second_moment_observe_count == 0) * 1
    p_hat_matrix = second_moment_observe_count

    T = second_moment_observe_M / p_hat_matrix

    diag_mask = torch.eye(d2).to(device) * T_masks
    off_diag_mask = (1 - diag_mask)*T_masks
    normalized_MTM_diag = torch.diag(normalized_MTM).mean()
    normalized_MTM_off_diag = normalized_MTM[off_diag_mask.bool()].mean()

print(d1*p, d1*p**2)
print(p_hat_count/100)
print("normalized_MTM_diag", normalized_MTM_diag)
print("normalized_MTM_off_diag", normalized_MTM_off_diag)
print(normalized_MTM)

print("T_diag", T[diag_mask.bool()].mean())
print("T_off_diag", T[off_diag_mask.bool()].mean())


50.0 0.25
tensor([[0.4700, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4600, 0.0100,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0100, 0.4400,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.4700, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.2800, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.5400]],
       device='cuda:1')
normalized_MTM_diag tensor(0.0040, device='cuda:1')
normalized_MTM_off_diag tensor(0.0040, device='cuda:1')
tensor([[0.0040, 0.0040, 0.0040,  ..., 0.0040, 0.0040, 0.0040],
        [0.0040, 0.0040, 0.0040,  ..., 0.0040, 0.0040, 0.0040],
        [0.0040, 0.0040, 0.0040,  ..., 0.0040, 0.0040, 0.0040],
        ...,
        [0.0040, 0.0040, 0.0040,  ..., 0.0040, 0.0040, 0.0040],
        [0.0040, 0.0040, 0.0040,  ..., 0.0040, 0.0040, 0.0040],
        [0.0040, 0.0040, 0.0040,  ..., 0.0040, 0.0040, 0.0040]],
       device='cuda:1')
T_diag tensor(0.0040, device='cuda:1')

In [7]:
print(T)

U, S, Vt = torch.linalg.svd(T)
S[r:]=0
T_svd = U @ torch.diag(S) @ Vt.t()
print(T_svd)
T_svd = T * T_masks.float() + T_svd * (1 - T_masks.float())
err_svd = torch.norm(normalized_MTM - T_svd) / torch.norm(normalized_MTM)
print(err_svd)

tensor([[0.0040, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0040, 0.0040,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0040, 0.0040,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.0040, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0040, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0040]],
       device='cuda:1')
tensor([[ 3.1938e-04, -5.9509e-04, -7.3510e-04,  ..., -1.6700e-03,
          1.3248e-03,  1.2041e-03],
        [ 1.1601e-03, -1.3429e-04, -5.4899e-05,  ..., -1.0345e-03,
          9.6819e-04,  2.2388e-03],
        [ 1.3450e-03, -1.0266e-03, -2.9463e-04,  ..., -1.2030e-04,
          8.8688e-04,  1.3036e-03],
        ...,
        [ 9.1879e-04, -5.2384e-05, -8.7361e-04,  ..., -2.1488e-03,
          6.6136e-04,  2.3233e-03],
        [ 6.5948e-04, -3.3019e-04, -3.7325e-04,  ..., -9.3498e-04,
          7.7319e-04,  1.1808e-03],
        [ 5.0186e-04, -3.1503e-04, -1.2028e-03,  .

In [8]:
print(d1*p**2)
diag_mask_1d = torch.rand(d2) > (d1*p**2)
diag_mask_2d = torch.diag(diag_mask_1d).to(device)
T_subsample_diag = T - T * diag_mask_2d.float()
print(T_subsample_diag)
print(T_subsample_diag.diag().nonzero().shape)

U, S, Vt = torch.linalg.svd(T_subsample_diag)
S[r:]=0
T_subsample_svd = U @ torch.diag(S) @ Vt.t()
T_subsample_svd_rescale = T_subsample_svd / (d1 * p**2)
T_subsample_svd_rescale = T * T_masks.float() + T_subsample_svd_rescale * (1 - T_masks.float())
print(T_subsample_svd_rescale)
err_subsample = torch.norm(T_subsample_svd_rescale - normalized_MTM) / torch.norm(normalized_MTM)
print(err_subsample)

0.25
tensor([[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0040,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0040, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0040, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],
       device='cuda:1')
torch.Size([258, 1])
tensor([[ 0.0040,  0.0003, -0.0046,  ..., -0.0005,  0.0027, -0.0030],
        [ 0.0020,  0.0040,  0.0040,  ...,  0.0026, -0.0009, -0.0022],
        [ 0.0037,  0.0040,  0.0040,  ...,  0.0043,  0.0017, -0.0011],
        ...,
        [ 0.0045,  0.0025, -0.0046,  ...,  0.0040,  0.0019, -0.0053],
        [ 0.0022,  0.0005, -0.0021,  ...,  0.0012,  0.0040, -0.0023],
        [ 0.0039,  0.0026, -0.0023,  ...,  0.0014,  0.0012,  0.0040]],
       device='cuda:1')
tensor(1.2225, device='cuda:1')


In [15]:
iipw = IIPW(M, observed_M, masks, r=r)
T_iipw = iipw.impute_grad(n_iter=500, lr=0.1)
print(T_iipw)
err_iipw = torch.norm(T_iipw - normalized_MTM) / torch.norm(normalized_MTM)
print(err_iipw)

Imputing...


 17%|█▋        | 86/500 [00:00<00:00, 856.74it/s]

tensor(1.0271, device='cuda:1')
tensor(0.8533, device='cuda:1')
tensor(0.7476, device='cuda:1')
tensor(0.6766, device='cuda:1')
tensor(0.6253, device='cuda:1')
tensor(0.5861, device='cuda:1')
tensor(0.5547, device='cuda:1')
tensor(0.5287, device='cuda:1')
tensor(0.5065, device='cuda:1')
tensor(0.4872, device='cuda:1')
tensor(0.4700, device='cuda:1')
tensor(0.4546, device='cuda:1')
tensor(0.4405, device='cuda:1')
tensor(0.4276, device='cuda:1')
tensor(0.4157, device='cuda:1')
tensor(0.4047, device='cuda:1')
tensor(0.3943, device='cuda:1')
tensor(0.3846, device='cuda:1')
tensor(0.3755, device='cuda:1')
tensor(0.3669, device='cuda:1')
tensor(0.3588, device='cuda:1')
tensor(0.3511, device='cuda:1')
tensor(0.3438, device='cuda:1')
tensor(0.3368, device='cuda:1')
tensor(0.3301, device='cuda:1')
tensor(0.3238, device='cuda:1')
tensor(0.3177, device='cuda:1')
tensor(0.3119, device='cuda:1')
tensor(0.3063, device='cuda:1')
tensor(0.3010, device='cuda:1')
tensor(0.2958, device='cuda:1')
tensor(0

 36%|███▌      | 180/500 [00:00<00:00, 903.70it/s]

tensor(0.0912, device='cuda:1')
tensor(0.0908, device='cuda:1')
tensor(0.0904, device='cuda:1')
tensor(0.0900, device='cuda:1')
tensor(0.0896, device='cuda:1')


...

tensor(0.0892, device='cuda:1')
tensor(0.0888, device='cuda:1')
tensor(0.0884, device='cuda:1')
tensor(0.0880, device='cuda:1')
tensor(0.0877, device='cuda:1')
tensor(0.0873, device='cuda:1')
tensor(0.0869, device='cuda:1')
tensor(0.0865, device='cuda:1')
tensor(0.0862, device='cuda:1')
tensor(0.0858, device='cuda:1')
tensor(0.0854, device='cuda:1')
tensor(0.0851, device='cuda:1')
tensor(0.0847, device='cuda:1')
tensor(0.0844, device='cuda:1')
tensor(0.0840, device='cuda:1')
tensor(0.0837, device='cuda:1')
tensor(0.0833, device='cuda:1')
tensor(0.0830, device='cuda:1')
tensor(0.0827, device='cuda:1')
tensor(0.0823, device='cuda:1')
tensor(0.0820, device='cuda:1')
tensor(0.0817, device='cuda:1')
tensor(0.0813, device='cuda:1')
tensor(0.0810, device='cuda:1')
tensor(0.0807, device='cuda:1')
tensor(0.0804, device='cuda:1')
tensor(0.0801, device='cuda:1')
tensor(0.0797, device='cuda:1')
tensor(0.0794, device='cuda:1')
tensor(0.0791, device='cuda:1')
tensor(0.0788, device='cuda:1')
tensor(0

 81%|████████▏ | 407/500 [00:00<00:00, 1061.44it/s]

tensor(0.0458, device='cuda:1')
tensor(0.0457, device='cuda:1')
tensor(0.0456, device='cuda:1')
tensor(0.0455, device='cuda:1')
tensor(0.0454, device='cuda:1')
tensor(0.0453, device='cuda:1')


100%|██████████| 500/500 [00:00<00:00, 1037.58it/s]

tensor(0.0452, device='cuda:1')
tensor(0.0451, device='cuda:1')
tensor(0.0450, device='cuda:1')
tensor(0.0449, device='cuda:1')
tensor(0.0449, device='cuda:1')
tensor(0.0448, device='cuda:1')
tensor(0.0447, device='cuda:1')
tensor(0.0446, device='cuda:1')
tensor(0.0445, device='cuda:1')
tensor(0.0444, device='cuda:1')
tensor(0.0443, device='cuda:1')
tensor(0.0442, device='cuda:1')
tensor(0.0441, device='cuda:1')
tensor(0.0440, device='cuda:1')
tensor(0.0439, device='cuda:1')
tensor(0.0439, device='cuda:1')
tensor(0.0438, device='cuda:1')
tensor(0.0437, device='cuda:1')
tensor(0.0436, device='cuda:1')
tensor(0.0435, device='cuda:1')
tensor(0.0434, device='cuda:1')
tensor(0.0433, device='cuda:1')
tensor(0.0433, device='cuda:1')
tensor(0.0432, device='cuda:1')
tensor(0.0431, device='cuda:1')
tensor(0.0430, device='cuda:1')
tensor(0.0429, device='cuda:1')
tensor(0.0428, device='cuda:1')
tensor(0.0427, device='cuda:1')
tensor(0.0427, device='cuda:1')
tensor(0.0426, device='cuda:1')
tensor(0